In [10]:
import pandas as pd
import pyodbc
import json
from datetime import date, datetime

# Charger configuration JSON
with open("staging_config.json", encoding="utf-8") as f:
    config_list = json.load(f)

# Connexion SQL Server (adapter)
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=Staging_Test;"
    "UID=moeness;"
    "PWD=azerty"
)


In [11]:
def load_and_historize(conn, config):
    table = config["table"]
    csv_path = config["csv"]
    id_col = config["id_col"]
    compare_cols = config["compare_cols"]
    today = date.today()

    try:
        df = pd.read_csv(csv_path, parse_dates=["Date_Examen"] if "Date_Examen" in compare_cols else [])
    except Exception as e:
        print(f"❌ Erreur lecture CSV {csv_path} : {e}")
        return

    df = df.astype("object")
    try:
        df_db = pd.read_sql(f"SELECT * FROM {table} WHERE Actif = 1", conn)
    except Exception as e:
        print(f"❌ Erreur lecture table {table} : {e}")
        return

    cursor = conn.cursor()

    for _, row in df.iterrows():
        old = df_db[df_db[id_col] == row[id_col]]
        if old.empty:
            # 🔵 INSERT (nouvelle ligne)
            cols_sql = ", ".join([id_col] + compare_cols + ["Date_Debut", "Actif", "Type_Changement"])
            placeholders = ", ".join(["?"] * (1 + len(compare_cols) + 3))
            values = [row[id_col]] + [row[col] for col in compare_cols] + [today, 1, 'INSERT']

            cursor.execute(f'''
                INSERT INTO {table} ({cols_sql})
                VALUES ({placeholders})
            ''', *values)

        else:
            old_row = old.iloc[0]
            if any(str(row[col]) != str(old_row[col]) for col in compare_cols):
                # 🟠 UPDATE ancienne version
                cursor.execute(f'''
                    UPDATE {table}
                    SET Date_Fin = ?, Actif = 0, Type_Changement = 'UPDATE', Last_Modified = ?
                    WHERE {id_col} = ? AND Actif = 1
                ''', today, datetime.now(), row[id_col])

                # 🔄 INSERT nouvelle version
                cols_sql = ", ".join([id_col] + compare_cols + ["Date_Debut", "Actif", "Type_Changement"])
                placeholders = ", ".join(["?"] * (1 + len(compare_cols) + 3))
                values = [row[id_col]] + [row[col] for col in compare_cols] + [today, 1, 'UPDATE']

                cursor.execute(f'''
                    INSERT INTO {table} ({cols_sql})
                    VALUES ({placeholders})
                ''', *values)

    conn.commit()
    cursor.close()
    print(f"✔️ Table {table} traitée avec succès.")


In [12]:
for config in config_list:
    print(f"📥 Traitement de {config['table']}...")
    load_and_historize(conn, config)


📥 Traitement de Joueurs...


C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_21084\3119216118.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql(f"SELECT * FROM {table} WHERE Actif = 1", conn)


✔️ Table Joueurs traitée avec succès.
📥 Traitement de Matchs...


C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_21084\3119216118.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql(f"SELECT * FROM {table} WHERE Actif = 1", conn)


✔️ Table Matchs traitée avec succès.
📥 Traitement de Donnees_Athletiques...


...

✔️ Table Examen_Locomoteur traitée avec succès.
📥 Traitement de Examen_ORL...


C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_21084\3119216118.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql(f"SELECT * FROM {table} WHERE Actif = 1", conn)


✔️ Table Examen_ORL traitée avec succès.
📥 Traitement de Examen_Stomatologique...


C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_21084\3119216118.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql(f"SELECT * FROM {table} WHERE Actif = 1", conn)


✔️ Table Examen_Stomatologique traitée avec succès.
